# LLM Classification su dataset PWC Multiclass (LM Studio)
Questo notebook implementa un classificatore basato su un LLM in esecuzione locale tramite LM Studio (modello "mlx-community/Ministral 3 3B Instruct 2512"). L'obiettivo è ri-classificare o testare la robustezza di una sottosezione del dataset `pwc_ai_multiclass_balanced.csv`.

In [2]:
import pandas as pd
import json
import time
from openai import OpenAI

# Inizializzazione del client OpenAI per puntare a LM Studio
# LM Studio, di default, espone le API compatibili con OpenAI su http://localhost:1234/v1
client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

In [3]:
# Test di connessione per verificare quale modello è caricato
try:
    models = client.models.list()
    print("Modelli disponibili:")
    for m in models.data:
        print(f"- {m.id}")
    SUPPORTED_MODEL = models.data[0].id if models.data else "ministral-3-3b-instruct-2512"
except Exception as e:
    print(f"Errore di connessione a LM Studio: {e}")
    SUPPORTED_MODEL = "ministral-3-3b-instruct-2512"

Modelli disponibili:
- ministral-3-3b-instruct-2512
- qwen/qwen3-vl-8b
- text-embedding-nomic-embed-text-v1.5


In [4]:
# Caricamento Dataset (es. prendiamo un campione di 50 record casuali per il test)
df_full = pd.read_csv('../data/processed/pwc_ai_multiclass_balanced.csv')

# Campioniamo i record per il test
df_test = df_full.sample(n=50, random_state=42).copy()
print(f"Dataset di test: {df_test.shape}")
display(df_test.head(3))

Dataset di test: (50, 3)


,description,label,desc_length
75721,several prior studies have suggested that word...,Research,1539.0
80184,this paper addresses an optimal control proble...,Robotics & Industry,1338.0
19864,we construct a physically-parameterized probab...,Data Science,1692.0


## Prompt Engineering per il modello Ministral 3B
Modelli più piccoli (3B) necessitano di istruzioni molto precise e vincoli rigorosi di formattazione. Il prompt richiederà specificamente l'output del solo nome della categoria, senza discorsi o preamboli.

In [5]:
ALLOWED_CATEGORIES = [
    "Automotive & UVs", "Data Science", "Enterprise", "Environment", 
    "Fintech", "Healthcare", "Media & Entertainment", "Research", 
    "Robotics & Industry", "Virtual Assistants"
]

def create_prompt(description):
    system_prompt = f"""You are an expert AI data annotator. Your task is to classify scientific research papers abstracts into EXACTLY ONE of the following highly specific categories:
{json.dumps(ALLOWED_CATEGORIES, indent=2)}

Rules:
1. You MUST read the description and choose the absolute best matching category.
2. If it is generic AI theory or academic terminology without an exact industry applied scope, classify it as "Research".
3. Reply ONLY with the exact text of the category name. Do not output anything else. No markdown, no JSON, no explanation."""

    user_prompt = f"Description to classify: {description}\n\nCategory name only:"
    
    return system_prompt, user_prompt

In [6]:
# Funzione di classificazione robusta
def classify_record(description, max_retries=3):
    system_p, user_p = create_prompt(description)
    
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=SUPPORTED_MODEL,
                messages=[
                    {"role": "system", "content": system_p},
                    {"role": "user", "content": user_p}
                ],
                temperature=0.0, # Temperatura 0 per consistenza e determinismo
                max_tokens=30 # Sufficiente per i 1-3 token del nome della categoria
            )
            
            result = response.choices[0].message.content.strip().strip('\'"').replace('\n', '')
            
            # Post-processing di pulizia per modelli chiacchieroni
            for cat in ALLOWED_CATEGORIES:
                if cat.lower() == result.lower() or cat.lower() in result.lower():
                    return cat
                    
            # Se la risposta non matcha categoricamente ma ci ha provato:
            return f"UNKNOWN_{result}"
            
        except Exception as e:
            print(f"Errore al tentativo {attempt + 1}: {e}")
            time.sleep(2)
            
    return "ERROR_MAX_RETRIES"

In [7]:
import os

# Esecuzione dell'intero dataset (o sottoinsieme) a batch per risparmiare RAM
BATCH_SIZE = 50
out_path = '../data/processed/llm_predictions_partial.csv'

# Decidiamo se usare df_full o df_test (qui df_test per provare, cambia in df_full per tutto)
df_target = df_test 

if os.path.exists(out_path):
    processed_df = pd.read_csv(out_path)
    processed_count = len(processed_df)
    print(f"Trovati {processed_count} record già processati. Riprendo...")
else:
    processed_count = 0
    # Inizializza il file con gli header
    pd.DataFrame(columns=list(df_target.columns) + ['llm_label']).to_csv(out_path, index=False)

print("Classificazione in batch in corso...")
df_to_process = df_target.iloc[processed_count:]
batch_records = []

for i, (index, row) in enumerate(df_to_process.iterrows()):
    idx_num = processed_count + i + 1
    short_desc = str(row['description'])[:1500] 
    
    pred_label = classify_record(short_desc)
    
    row_dict = row.to_dict()
    row_dict['llm_label'] = pred_label
    batch_records.append(row_dict)
    
    print(f"[{idx_num}/{len(df_target)}] TRUE: {row['label']:<25} | PRED: {pred_label}")
    
    # Salva su disco e svuota la RAM ogni BATCH_SIZE
    if len(batch_records) >= BATCH_SIZE:
        pd.DataFrame(batch_records).to_csv(out_path, mode='a', header=False, index=False)
        batch_records = []

# Salva l'ultimo batch residuo
if len(batch_records) > 0:
    pd.DataFrame(batch_records).to_csv(out_path, mode='a', header=False, index=False)
    batch_records = []

print(f"Classificazione batch completata e salvata in {out_path}")

Classificazione in batch in corso...
[1/50] TRUE: Research                  | PRED: Research
[2/50] TRUE: Robotics & Industry       | PRED: Robotics & Industry
[3/50] TRUE: Data Science              | PRED: Research
[4/50] TRUE: Research                  | PRED: Research
[5/50] TRUE: Virtual Assistants        | PRED: Research
[6/50] TRUE: Research                  | PRED: Research
[7/50] TRUE: Robotics & Industry       | PRED: Robotics & Industry
[8/50] TRUE: Robotics & Industry       | PRED: Robotics & Industry
[9/50] TRUE: Media & Entertainment     | PRED: Robotics & Industry
[10/50] TRUE: Healthcare                | PRED: Healthcare
[11/50] TRUE: Enterprise                | PRED: Research
[12/50] TRUE: Fintech                   | PRED: Robotics & Industry
[13/50] TRUE: Fintech                   | PRED: Research
[14/50] TRUE: Data Science              | PRED: Research
[15/50] TRUE: Enterprise                | PRED: Robotics & Industry
[16/50] TRUE: Robotics & Industry       | PRED: R

In [8]:
# Ricarichiamo dal file batch salvato per la valutazione finale
df_results = pd.read_csv(out_path)

# Analisi dei risultati (accuratezza locale sul ground truth)
matched = (df_results['label'] == df_results['llm_label']).sum()
total = len(df_results)
print(f"\nMatch totali: {matched}/{total} ({matched/total*100:.2f}%)")

# Verifica record sconosciuti / malformattati
unknown_preds = df_results[df_results['llm_label'].str.startswith('UNKNOWN_')]
print(f"Errori di formattazione output: {len(unknown_preds)}")
if len(unknown_preds) > 0:
    display(unknown_preds[['description', 'llm_label']].head())


Match totali: 17/50 (34.00%)
Errori di formattazione output: 0
